# LIME for Tabular and Text Data
Explore model explanations on tabular (Iris) and text (20 Newsgroups) datasets.
Images are saved in a local `out/` folder for easy download.

### Task Description
- **Tabular**: Explain Iris classifications with Random Forest.
- **Text**: Explain text classifications with Logistic Regression on 20 Newsgroups.
- View the **full document** for text examples and download explanation images.

In [1]:
from pathlib import Path

OUT_DIR = Path("out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def out_path(filename):
    return OUT_DIR / filename


In [2]:
# Uncomment to install required packages:
# !pip install lime scikit-learn ipywidgets

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

from sklearn.datasets import load_iris, fetch_20newsgroups
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from lime.lime_tabular import LimeTabularExplainer
from lime.lime_text import LimeTextExplainer



## 1. Tabular Data: Iris Classification

In [3]:
# Load Iris data
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

# Train Random Forest classifier
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X, y)

# Initialize LIME tabular explainer
tabular_explainer = LimeTabularExplainer(
    X, feature_names=feature_names, class_names=class_names,
    discretize_continuous=True, random_state=0
)


In [4]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ipywidgets import interact, IntSlider

ROOT = Path.cwd()
while not (ROOT / "xai_book").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai_book.plotting import (
    BACKGROUND,
    BLUE as BOOK_BLUE,
    CHAPTER_GRAY,
    GRID as LIGHT_GRAY,
    ORANGE as BOOK_ORANGE,
    apply_notebook_style,
)

apply_notebook_style()

book_colors = {
    'blue': BOOK_BLUE,
    'orange': BOOK_ORANGE,
    'light_gray': LIGHT_GRAY,
    'dark_gray': CHAPTER_GRAY,
    'background': BACKGROUND,
}

def explain_tabular(idx):
    exp = tabular_explainer.explain_instance(
        X[idx], rf.predict_proba, num_features=4, top_labels=1
    )
    label = exp.top_labels[0]

    fig = exp.as_pyplot_figure(label=label)
    ax = fig.axes[0]

    fig.patch.set_facecolor(book_colors['background'])
    ax.set_facecolor(book_colors['background'])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False)
    ax.set_title(
        f'LIME Explanation (Instance {idx}, Label={label})',
        color=book_colors['blue'],
        fontsize=14,
        pad=12,
    )

    for bar in ax.patches:
        weight = bar.get_width()
        bar.set_color(book_colors['orange'] if weight > 0 else book_colors['blue'])

    plt.tight_layout()
    png_fname = out_path(f"lime_tabular_{idx}.png")
    pdf_fname = out_path(f"lime_tabular_{idx}.pdf")
    fig.savefig(png_fname, bbox_inches='tight', dpi=300)
    fig.savefig(pdf_fname, bbox_inches='tight')
    plt.show()
    print('Saved:', png_fname, 'and', pdf_fname)

    contribs = exp.as_list(label=label)
    df = pd.DataFrame(contribs, columns=['feature', 'weight'])
    print('\nFeature contributions:')
    display(df)

    pred = rf.predict(X[idx].reshape(1, -1))[0]
    probs = rf.predict_proba(X[idx].reshape(1, -1))[0]
    print(f"True class:      {class_names[y[idx]]}")
    print(f"Predicted class: {class_names[pred]}")
    print('Probabilities:   ', dict(zip(class_names, np.round(probs, 3))))

interact(
    explain_tabular,
    idx=IntSlider(min=0, max=X.shape[0] - 1, step=1, value=0, description='Instance'),
)


interactive(children=(IntSlider(value=0, description='Instance', max=149), Output()), _dom_classes=('widget-in…

<function __main__.explain_tabular(idx)>

## 2. Text Data: 20 Newsgroups Classification

In [5]:
# Load text data (two categories)
categories = ['sci.space', 'rec.autos']
newsgroups = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes')
)
X_text, y_text = newsgroups.data, newsgroups.target
target_names = newsgroups.target_names

# Train text classification pipeline
pipeline = make_pipeline(
    TfidfVectorizer(lowercase=True, stop_words='english'),
    LogisticRegression(max_iter=10000)
)
pipeline.fit(X_text, y_text)

# Initialize LIME text explainer
text_explainer = LimeTextExplainer(class_names=target_names)


In [6]:
from IPython.display import display, FileLink

def explain_text(idx):
    doc = X_text[idx]
    pred = pipeline.predict([doc])[0]
    exp = text_explainer.explain_instance(
        doc, pipeline.predict_proba, labels=[pred], num_features=6
    )

    fig = exp.as_pyplot_figure(label=pred)
    ax = fig.axes[0]
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False)
    ax.set_title(
        f'Explanation (idx={idx}, label={pred})',
        color=book_colors['blue'],
        fontsize=14,
        pad=12,
    )
    for bar in ax.patches:
        weight = bar.get_width()
        bar.set_color(book_colors['orange'] if weight > 0 else book_colors['blue'])

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    pdf_fname = out_path(f"lime_text_{idx}.pdf")
    fig.savefig(pdf_fname, bbox_inches='tight')

    true_lbl = target_names[y_text[idx]]
    overlay = f"True: {true_lbl}   ▶   Pred: {target_names[pred]}"
    fig.text(
        0.5,
        0.95,
        overlay,
        ha='center',
        va='top',
        fontsize=12,
        color=book_colors['dark_gray'],
        bbox=dict(
            boxstyle='round,pad=0.3',
            facecolor='white',
            edgecolor=book_colors['light_gray'],
        ),
    )

    png_fname = out_path(f"lime_text_{idx}.png")
    fig.savefig(png_fname, bbox_inches='tight', dpi=300)
    plt.show()

    print('Saved PDF (no overlay):', pdf_fname)
    display(FileLink(str(pdf_fname)))

    contribs = exp.as_list(label=pred)
    df = pd.DataFrame(contribs, columns=['word', 'weight'])
    print('\nWord contributions:')
    display(df)

interact(
    explain_text,
    idx=IntSlider(min=0, max=len(X_text) - 1, step=1, value=0, description='Instance'),
)


interactive(children=(IntSlider(value=0, description='Instance', max=1186), Output()), _dom_classes=('widget-i…

<function __main__.explain_text(idx)>

In [7]:
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, FileLink
from ipywidgets import interact, IntSlider

ROOT = Path.cwd()
while not (ROOT / "xai_book").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai_book.plotting import (
    BACKGROUND,
    BLUE as BOOK_BLUE,
    CHAPTER_GRAY,
    GRID as LIGHT_GRAY,
    ORANGE as BOOK_ORANGE,
    apply_notebook_style,
)

apply_notebook_style()

book_colors = {
    'blue': BOOK_BLUE,
    'orange': BOOK_ORANGE,
    'light_gray': LIGHT_GRAY,
    'dark_gray': CHAPTER_GRAY,
    'background': BACKGROUND,
}

def explain_text(idx):
    doc = X_text[idx]
    pred = pipeline.predict([doc])[0]
    exp = text_explainer.explain_instance(
        doc, pipeline.predict_proba, labels=[pred], num_features=6
    )
    contribs = exp.as_list(label=pred)
    words, weights = zip(*contribs)

    fig = plt.figure(figsize=(12, 6))
    ax_text = fig.add_subplot(1, 2, 1)
    ax_text.axis('off')
    wrapped = '\n'.join(textwrap.wrap(doc, width=60))
    ax_text.text(
        0,
        1,
        wrapped,
        va='top',
        ha='left',
        fontsize=10,
        color=book_colors['dark_gray'],
    )
    ax_text.set_title('Document', color=book_colors['blue'], pad=10)

    ax_bar = fig.add_subplot(1, 2, 2)
    y_pos = range(len(words))
    colors = [book_colors['orange'] if weight > 0 else book_colors['blue'] for weight in weights]
    ax_bar.barh(y_pos, weights, color=colors)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(words, fontsize=10)
    ax_bar.invert_yaxis()
    ax_bar.axvline(0, color=book_colors['light_gray'])
    for spine in ax_bar.spines.values():
        spine.set_visible(False)
    ax_bar.tick_params(left=False, bottom=False)
    ax_bar.set_title('LIME word weights', color=book_colors['blue'], pad=10)

    pdf_fname = out_path(f"lime_text_{idx}.pdf")
    plt.tight_layout()
    fig.savefig(pdf_fname, bbox_inches='tight')

    overlay = f"True: {target_names[y_text[idx]]}   ▶   Pred: {target_names[pred]}"
    fig.text(
        0.5,
        0.95,
        overlay,
        ha='center',
        va='top',
        fontsize=12,
        color=book_colors['dark_gray'],
        bbox=dict(
            boxstyle='round,pad=0.3',
            facecolor='white',
            edgecolor=book_colors['light_gray'],
        ),
    )

    png_fname = out_path(f"lime_text_{idx}.png")
    fig.savefig(png_fname, bbox_inches='tight', dpi=300)
    plt.show()

    print('Saved PDF (no overlay):', pdf_fname)
    display(FileLink(str(pdf_fname)))

    df = pd.DataFrame(contribs, columns=['word', 'weight'])
    print('\nWord contributions:')
    display(df)

interact(
    explain_text,
    idx=IntSlider(min=0, max=len(X_text) - 1, step=1, value=0, description='Instance'),
)


interactive(children=(IntSlider(value=0, description='Instance', max=1186), Output()), _dom_classes=('widget-i…

<function __main__.explain_text(idx)>